# 第8回 線形フィルタと時間周波数解析

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：重みをかけて足すだけの操作は、系列のどの周期を残し、どの周期を消すのか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "WaveletComp"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月と曜日の名前を英語にする

### A-1. 2つの操作をかけてみる

第2回と同じ人工の系列を使います。長さは364日で、「ゆっくりした傾向 ＋ 7日でくり返す上下 ＋ ノイズ」でできています。

In [ ]:
set.seed(1)
n <- 364
day <- 1:n
trend  <- 100 + 8 * sin(2 * pi * day / 364) + 0.02 * day    # ゆっくりした傾向
season <- rep(c(5, 6, 6, 5, 4, -12, -14), length.out = n)   # 7日でくり返す上下（合計は0）
noise  <- rnorm(n, sd = 2)                                  # ノイズ
x <- trend + season + noise

この系列に、これまでの回で使った2つの操作をかけます。

- 移動平均（第2回）：前後の値を平均する。
- 差分（第4回）：1つ前の値を引く。

どちらも、系列の値に重みをかけて足す操作です。このような操作を、線形フィルタ（linear filter）と呼びます。

In [ ]:
# 移動平均の重み（長さ m。m が偶数のときは、両端の重みを半分にして左右対称にする）
ma_weights <- function(m) {
  if (m %% 2 == 1) rep(1 / m, m) else c(0.5, rep(1, m - 1), 0.5) / m
}
# 重みのまん中をラグ0としたときの、各重みのラグ
centered_lags <- function(a) seq_along(a) - (length(a) + 1) / 2
# 移動平均をかける（前後の値を使う。両端は計算できないので NA）
apply_ma <- function(x, m) as.numeric(stats::filter(x, ma_weights(m), sides = 2))
# 差分をとる（lag 時点前の値を引く。最初の lag 個は NA）
apply_diff <- function(x, lag = 1) c(rep(NA, lag), diff(x, lag = lag))

In [ ]:
out <- rbind(
  data.frame(day = day, panel = "1 series", value = x),
  data.frame(day = day, panel = "2 moving average (7 days)", value = apply_ma(x, 7)),
  data.frame(day = day, panel = "3 difference", value = apply_diff(x))
)
ggplot(out, aes(day, value)) +
  geom_line(na.rm = TRUE) +
  facet_wrap(~panel, ncol = 1, scales = "free_y") +
  labs(x = "Day", y = "Value")

- 移動平均：7日でくり返す上下が消え、ゆっくりした傾向が残ります。
- 差分：ゆっくりした傾向が消え、7日でくり返す上下とノイズが残ります。

残るものと消えるものが、逆になっています。この違いを、周期の言葉で説明するのが今日の目標です。

### A-2. 波を入れると、同じ周期の波が出る

フィルタに、周期のはっきりした波（余弦波）を1本だけ入れます。周期は28日、10日、7日の3通りです。フィルタは7日の移動平均です。

In [ ]:
t <- 1:84
sines <- do.call(rbind, lapply(c(28, 10, 7), function(p) {
  input <- cos(2 * pi * t / p)
  data.frame(t = t, period = sprintf("period %d days", p),
             input = input, output = apply_ma(input, 7))
}))
sines$period <- factor(sines$period, levels = unique(sines$period))
ggplot(sines, aes(t)) +
  geom_line(aes(y = input), colour = "grey60") +
  geom_line(aes(y = output), colour = "red", na.rm = TRUE) +
  facet_wrap(~period, ncol = 1) +
  labs(x = "Day", y = "Value (grey: input, red: output)")

- 出てくる系列（赤）は、入れた波（灰色）と同じ周期の波です。変わるのは、波の高さ（振幅）だけです。
- 周期28日の波は、ほぼそのまま通ります。周期10日の波は小さくなり、周期7日の波は完全に消えます。

「出た波の振幅 ÷ 入れた波の振幅」を、ゲイン（gain）と呼びます。
周波数（1時点あたりに波が何回くり返すか。周期の逆数）$\omega$ ごとにゲインを並べたものが、フィルタの性質を表します。

重み $a_j$ のフィルタ $y_t = \sum_j a_j x_{t-j}$ に対して、次の $H(\omega)$ を周波数応答（frequency response）と呼びます。ゲインは、その絶対値 $|H(\omega)|$ です。

$$
H(\omega) = \sum_j a_j e^{-2\pi i \omega j}
$$

In [ ]:
# 周波数応答 H(omega)：重み a と、それぞれのラグ lags から計算する
freq_response <- function(a, lags, omega) {
  sapply(omega, function(w) sum(a * exp(-2i * pi * w * lags)))
}
# ゲイン |H(omega)|
gain <- function(a, lags, omega) Mod(freq_response(a, lags, omega))

式から計算したゲイン（線）を、実際に波を入れて測ったゲイン（点）と比べます。
測るときは、周波数 $\omega$ の波を入れ、出てきた系列に同じ周波数の cos と sin を回帰で当てはめて、振幅を出します。

In [ ]:
# 波を実際に入れて、ゲインを測る（f はフィルタをかける関数）
measure_gain <- function(f, omega, n = 280) {
  t <- 1:n
  sapply(omega, function(w) {
    out <- f(cos(2 * pi * w * t))                               # 振幅1の波を入れる
    fit <- lm(out ~ cos(2 * pi * w * t) + sin(2 * pi * w * t))  # 出てきた波の振幅を測る
    sqrt(sum(coef(fit)[2:3]^2))
  })
}
omega_test <- seq(0.02, 0.48, by = 0.02)    # 波を入れて測る周波数
omega <- seq(0, 0.5, length.out = 501)      # 式で計算する周波数
measured <- rbind(
  data.frame(filter = "moving average (7 days)", omega = omega_test,
             gain = measure_gain(function(x) apply_ma(x, 7), omega_test)),
  data.frame(filter = "difference", omega = omega_test,
             gain = measure_gain(apply_diff, omega_test))
)
formula <- rbind(
  data.frame(filter = "moving average (7 days)", omega = omega,
             gain = gain(ma_weights(7), centered_lags(ma_weights(7)), omega)),
  data.frame(filter = "difference", omega = omega,
             gain = gain(c(1, -1), c(0, 1), omega))
)
ggplot(formula, aes(omega, gain)) +
  geom_line() +
  geom_point(data = measured, colour = "red") +
  facet_wrap(~filter, scales = "free_y") +
  labs(x = "Frequency (cycles per day)", y = "Gain (line: formula, red: measured)")

点は、線の上に乗ります。

- 移動平均：低い周波数（ゆっくりした波）でゲインが1に近く、高い周波数で小さくなります。低い周波数を通すので、低域通過フィルタ（low-pass filter）と呼びます。
- 差分：周波数0でゲインが0、周波数が上がるほど大きくなり、最も高い周波数0.5（周期2日）で2になります。高い周波数を通すので、高域通過フィルタ（high-pass filter）と呼びます。差分は、高い周波数を通すだけでなく、2倍まで強めます。

### A-3. 移動平均と差分のゲイン

移動平均の長さを5日、7日、28日と変えて、ゲインを比べます。縦の破線は、周期7日（周波数1/7）と、その2倍、3倍の周波数です。

In [ ]:
ma_gain <- do.call(rbind, lapply(c(5, 7, 28), function(m) {
  a <- ma_weights(m)
  data.frame(m = sprintf("%2d days", m), omega = omega, gain = gain(a, centered_lags(a), omega))
}))
ggplot(ma_gain, aes(omega, gain, colour = m)) +
  geom_vline(xintercept = (1:3) / 7, linetype = "dashed", colour = "grey50") +
  geom_line() +
  labs(x = "Frequency (cycles per day)", y = "Gain", colour = "Length")
# 周波数 1/7, 2/7, 3/7 でのゲイン
sapply(c(5, 7, 28), function(m) {
  a <- ma_weights(m)
  round(gain(a, centered_lags(a), (1:3) / 7), 3)
})

- 7日の移動平均は、周波数1/7、2/7、3/7でゲインがちょうど0です。7日でくり返す上下は、どんな形でもこの3つの周波数の波の和で書けるので、すべて消えます。
- 5日の移動平均では、周波数1/7のゲインが0になりません。7日でくり返す上下が残ります。
- 28日の移動平均も、1/7、2/7、3/7で0になります。平均する長さが長いほど、通る周波数の範囲は狭くなります。

次は差分です。1階の差分、季節差分（7日前の値を引く）、その両方、の3つを比べます。
両方をかけたフィルタは、第4回の季節ARIMAモデルで $d = 1$、$D = 1$ としたときの差分です。

In [ ]:
g_diff1 <- gain(c(1, -1), c(0, 1), omega)             # 差分
g_diff7 <- gain(c(1, -1), c(0, 7), omega)             # 季節差分（7日）
g_both  <- gain(c(1, -1, -1, 1), c(0, 1, 7, 8), omega)   # 両方
diff_gain <- rbind(
  data.frame(filter = "1 difference", omega = omega, gain = g_diff1),
  data.frame(filter = "2 seasonal difference (7 days)", omega = omega, gain = g_diff7),
  data.frame(filter = "3 both", omega = omega, gain = g_both)
)
ggplot(diff_gain, aes(omega, gain)) +
  geom_vline(xintercept = (1:3) / 7, linetype = "dashed", colour = "grey50") +
  geom_line() +
  facet_wrap(~filter, ncol = 1) +
  labs(x = "Frequency (cycles per day)", y = "Gain")
# 2つを続けてかけたときのゲインは、それぞれのゲインの積になる
all.equal(g_both, g_diff1 * g_diff7)

- 差分：周波数0（傾向）を消します。周波数1/7では0にならないので、7日でくり返す上下は残ります。
- 季節差分：周波数0と、1/7、2/7、3/7を消します。傾向も、7日でくり返す上下も消えます。そのあいだの周波数は、最大で2倍に強めます。
- フィルタを続けてかけると、ゲインは積になります。

移動平均は、前後の値を使いました。実際の場面では、まだ来ていない先の値は使えません。過去の値だけで平均すると、どうなるかを見ます。

In [ ]:
t <- 1:140
slow <- cos(2 * pi * t / 56)                                           # 周期56日の波
ma_two <- apply_ma(slow, 7)                                            # 前後3日ずつを使う
ma_one <- as.numeric(stats::filter(slow, rep(1 / 7, 7), sides = 1))    # 過去の7日だけを使う
sided <- rbind(data.frame(t = t, kind = "input", value = slow),
               data.frame(t = t, kind = "two-sided (past and future)", value = ma_two),
               data.frame(t = t, kind = "one-sided (past only)", value = ma_one))
ggplot(sided, aes(t, value, colour = kind)) +
  geom_line(na.rm = TRUE) +
  labs(x = "Day", y = "Value", colour = NULL)
# 過去だけを使う移動平均の遅れ（日）：周波数応答の偏角を、時間に直す
Arg(freq_response(rep(1 / 7, 7), 0:6, 1 / 56)) / (2 * pi / 56)

過去だけを使う移動平均は、波の高さは同じですが、3日遅れます。周波数応答 $H(\omega)$ の絶対値がゲイン、偏角が波のずれ（位相、phase）を表します。

### A-4. スペクトルは、ゲインの2乗倍になる

第7回で、系列を周波数ごとの強さに分けて見るピリオドグラム（periodogram）と、その期待値にあたるスペクトル密度（spectral density）を扱いました。
フィルタをかけると、スペクトル密度は、周波数ごとにゲインの2乗倍になります。

$$
f_y(\omega) = |H(\omega)|^2 f_x(\omega)
$$

ホワイトノイズ（分散1）で確かめます。ホワイトノイズのスペクトル密度は、どの周波数でも1です。
長さ256のホワイトノイズを500本作り、フィルタをかける前とかけた後のピリオドグラムを、それぞれ平均します。

In [ ]:
# ピリオドグラム：周波数 j/n（j = 1, 2, ...）ごとの、波の強さ（第7回と同じ関数）
pgram <- function(x) {
  n <- length(x)
  d <- fft(x - mean(x)) / sqrt(n)       # 離散フーリエ変換（平均を引いてから）
  j <- 1:floor((n - 1) / 2)             # 周波数が 0 より大きく、0.5 より小さいところだけを見る
  data.frame(freq = j / n, period = n / j, spec = Mod(d[j + 1])^2)
}

In [ ]:
set.seed(1)
keep <- 4:259                                   # フィルタの両端（NA）を除いた256個
one <- function() {
  w <- rnorm(262)                               # ホワイトノイズ（分散1）
  cbind(noise = pgram(w[keep])$spec,
        ma    = pgram(apply_ma(w, 7)[keep])$spec,
        diff  = pgram(apply_diff(w)[keep])$spec)
}
avg <- Reduce(`+`, replicate(500, one(), simplify = FALSE)) / 500   # 500本の平均
freq <- pgram(rnorm(256))$freq
spec <- rbind(
  data.frame(series = "1 white noise", freq = freq, sim = avg[, "noise"], theory = 1),
  data.frame(series = "2 moving average (7 days)", freq = freq, sim = avg[, "ma"],
             theory = gain(ma_weights(7), -3:3, freq)^2),
  data.frame(series = "3 difference", freq = freq, sim = avg[, "diff"],
             theory = gain(c(1, -1), c(0, 1), freq)^2)
)
ggplot(spec, aes(freq)) +
  geom_line(aes(y = sim), colour = "grey50") +
  geom_line(aes(y = theory), colour = "red") +
  facet_wrap(~series, ncol = 1, scales = "free_y") +
  labs(x = "Frequency (cycles per step)", y = "Power (grey: simulation, red: gain squared)")

平均したピリオドグラム（灰色）は、ゲインの2乗（赤）に重なります。

ホワイトノイズは、隣どうしが無相関です。フィルタをかけたあとはどうなるかを、分散とラグ1の自己相関で確かめます。

In [ ]:
set.seed(1)
w <- rnorm(10000)
stat <- function(v) {
  v <- v[!is.na(v)]
  c(variance = var(v), acf_lag1 = acf(v, lag.max = 1, plot = FALSE)$acf[2])
}
round(rbind(white_noise = stat(w),
            moving_average = stat(apply_ma(w, 7)),
            difference = stat(apply_diff(w))), 3)

- 移動平均：分散は約1/7に減りますが、ラグ1の自己相関が約6/7になります。隣どうしが、7個のうち6個の値を共有するからです。
- 差分：分散は約2倍に増え、ラグ1の自己相関は約−0.5になります。

iidの系列にフィルタをかけると、iidではなくなります。フィルタをかけた系列を、独立な観測として扱ってはいけません。

### A-5. 見たい周期だけを通すフィルタ

移動平均は低い周波数を、差分は高い周波数を通しました。まん中の、決まった周期だけを通すフィルタを、帯域通過フィルタ（band-pass filter）と呼びます。

重みを「波の形」にすると、その波の周期だけを通すフィルタになります。ここでは、周期 `period` の波に、山の形の窓（幅 `width` の正規分布の形）をかけたものを重みにします。
ここでの窓は、時間の軸で、どの範囲の値を使うかを決める重みです。第7回の窓関数（周波数の軸で、となりどうしを平均する重み）とは別のものです。

- cos の形の重みと、sin の形の重みの2組を使い、結果を複素数の実部と虚部に入れます。
- 実部が、取り出した成分です。絶対値が、その時点での成分の振幅です。

In [ ]:
# 波の形の重み：周期 period の波 × 幅 width（標準偏差）の山の形の窓
wave_weights <- function(period, width) {
  j <- -ceiling(3 * width):ceiling(3 * width)          # 窓は、幅の3倍まで
  g <- exp(-j^2 / (2 * width^2))                       # 山の形の窓
  list(lags = j, a = 2 * g / sum(g) * exp(2i * pi * j / period))
}
# 周期 period の成分を取り出す（実部が成分、絶対値が振幅。端は計算できないので NA）
band_component <- function(x, period, width = period) {
  a <- wave_weights(period, width)$a
  if (length(a) > length(x)) return(rep(NA_complex_, length(x)))   # 窓が系列より長い
  complex(real      = as.numeric(stats::filter(x, Re(a), sides = 2)),
          imaginary = as.numeric(stats::filter(x, Im(a), sides = 2)))
}

周期7日を取り出すフィルタ（実部を作る、cos の形の重み）のゲインを、窓の幅を変えて描きます。

In [ ]:
band_gain <- do.call(rbind, lapply(c(3.5, 7, 14), function(wd) {
  w <- wave_weights(7, wd)
  data.frame(width = sprintf("%4.1f days", wd), omega = omega,
             gain = gain(Re(w$a), w$lags, omega))      # 成分を取り出す、実部（cos の形）の重みのゲイン
}))
ggplot(band_gain, aes(omega, gain, colour = width)) +
  geom_vline(xintercept = 1 / 7, linetype = "dashed", colour = "grey50") +
  geom_line() +
  labs(x = "Frequency (cycles per day)", y = "Gain", colour = "Window width")

- ゲインは、周波数1/7のまわりだけで1に近く、離れると0になります。
- 窓の幅を広げると、通る周波数の範囲が狭くなります。窓の幅を狭めると、通る範囲が広がります。

A-1の系列から、周期7日の成分を取り出します。

In [ ]:
z <- band_component(x, period = 7, width = 14)
band <- data.frame(day = day, detrended = x - apply_ma(x, 7), component = Re(z), amplitude = Mod(z))
ggplot(band |> filter(day >= 100, day <= 160), aes(day)) +
  geom_line(aes(y = detrended), colour = "grey60") +
  geom_line(aes(y = component), colour = "red") +
  geom_line(aes(y = amplitude), colour = "blue", linetype = "dashed") +
  labs(x = "Day", y = "Value (red: 7-day component)")
round(mean(band$amplitude, na.rm = TRUE), 2)

- 灰色は、系列から7日の移動平均を引いたものです。取り出した成分（赤）は、周期7日のなめらかな波で、もとの上下（5日高く、2日低い）とは形が違います。周波数1/7の波だけを通し、2/7と3/7の波を通さないからです。
- 振幅（青の破線）は、ほぼ一定です。この系列は、7日の上下の大きさが変わらないように作ってあります。

### A-6. 周期が時間とともに変わる系列

周波数が時間とともに変わる系列を、チャープ信号（chirp signal）と呼びます。
周波数が0.02から0.2まで上がっていく系列と、それを時間の向きに逆さにした系列（周波数が下がっていく）を作ります。

In [ ]:
n <- 1024
t <- 1:n
freq_t <- 0.02 + (0.20 - 0.02) * t / n          # 時点 t での周波数（周期は50から5へ）
chirp_up   <- cos(2 * pi * cumsum(freq_t))      # 周波数が上がっていく
chirp_down <- rev(chirp_up)                     # 周波数が下がっていく
chirps <- rbind(data.frame(t = t, signal = "1 chirp up", value = chirp_up),
                data.frame(t = t, signal = "2 chirp down", value = chirp_down))
ggplot(chirps |> filter(t <= 400 | t > 624), aes(t, value)) +
  geom_line() +
  facet_grid(signal ~ ifelse(t <= 400, "first 400 steps", "last 400 steps"), scales = "free_x") +
  labs(x = "Time (step)", y = "Value")

2本のピリオドグラムを比べます。

In [ ]:
pg_chirp <- rbind(cbind(signal = "1 chirp up", pgram(chirp_up)),
                  cbind(signal = "2 chirp down", pgram(chirp_down)))
ggplot(pg_chirp, aes(freq, spec)) +
  geom_line() +
  facet_wrap(~signal, ncol = 1) +
  labs(x = "Frequency (cycles per step)", y = "Periodogram")
# 2本のピリオドグラムは同じか
all.equal(pgram(chirp_up)$spec, pgram(chirp_down)$spec)

2本のピリオドグラムは、まったく同じです。ピリオドグラムは「どの周波数が、全体でどれだけあるか」を教えますが、「いつ」あるかは教えません。

そこで、時刻ごとに、そのまわりだけを見て周波数を調べます。
A-5の帯域通過フィルタを、たくさんの周期について並べればできます。時刻を横、周波数（または周期）を縦にとり、振幅を色で表します。

- 窓の幅を、どの周期でも同じにする：短時間フーリエ変換（short-time Fourier transform, STFT）。図をスペクトログラム（spectrogram）と呼ぶ。
- 窓の幅を、周期に比例させる：ウェーブレット変換（wavelet transform）。図をスカログラム（scalogram）と呼ぶ。

In [ ]:
# 時刻 × 周期ごとの振幅の表（width_fun は、周期から窓の幅を決める関数）
tf_table <- function(x, periods, width_fun, time = seq_along(x)) {
  do.call(rbind, lapply(periods, function(p) {
    data.frame(time = time, period = p, amplitude = Mod(band_component(x, p, width_fun(p))))
  }))
}
# 図：横が時刻、縦が周期（対数の目盛り）か周波数、色が振幅。灰色は、端で計算できないところ
plot_tf <- function(d, axis = "period", breaks = waiver()) {
  g <- if (axis == "period") {
    ggplot(d, aes(time, period, fill = amplitude)) + scale_y_log10(breaks = breaks)
  } else {
    ggplot(d, aes(time, 1 / period, fill = amplitude))
  }
  g + geom_raster() +
    scale_fill_viridis_c(na.value = "grey85") +
    labs(x = "Time (step)", y = if (axis == "period") "Period (steps)" else "Frequency (cycles per step)",
         fill = "Amplitude")
}

2本のチャープ信号のスペクトログラムを描きます。窓の幅は20時点です。

In [ ]:
freqs <- seq(0.01, 0.25, by = 0.005)
stft_chirp <- rbind(
  cbind(signal = "1 chirp up", tf_table(chirp_up, 1 / freqs, function(p) 20)),
  cbind(signal = "2 chirp down", tf_table(chirp_down, 1 / freqs, function(p) 20))
)
plot_tf(stft_chirp, axis = "frequency") +
  facet_wrap(~signal, ncol = 1)

周波数が上がっていく様子と、下がっていく様子が、線として見えます。ピリオドグラムでは区別できなかった2本が、区別できます。

### A-7. 窓の幅と、2つの細かさ

窓の幅は、自由には決められません。別の系列で確かめます。長さは2048です。

- 周期128のゆっくりした波が、ずっと続く。
- 周期6の速い波が、1000時点目から30時点のあいだだけ現れる。

In [ ]:
n <- 2048
t <- 1:n
mix <- cos(2 * pi * t / 128) + ifelse(t >= 1000 & t < 1030, cos(2 * pi * t / 6), 0)
ggplot(data.frame(t = t, value = mix) |> filter(t >= 700, t <= 1300), aes(t, value)) +
  geom_line() +
  labs(x = "Time (step)", y = "Value")

窓の幅を3通りに変えて、同じ系列を見ます。

In [ ]:
periods <- 2^seq(2, 8, by = 0.125)          # 周期4から256まで（等比で49個）
tf_mix <- rbind(
  cbind(method = "1 STFT, width 4", tf_table(mix, periods, function(p) 4)),
  cbind(method = "2 STFT, width 64", tf_table(mix, periods, function(p) 64)),
  cbind(method = "3 wavelet, width = period", tf_table(mix, periods, function(p) p))
)
plot_tf(tf_mix, breaks = c(6, 16, 32, 64, 128)) +
  facet_wrap(~method, ncol = 1)

- 窓が狭い（幅4）：速い波が「いつ」現れたかは、はっきり分かります。ゆっくりした波は、周期128のところに集まらず、縦の縞になります。窓が波1回分よりずっと短いので、波の山と谷をそのまま拾うだけで、周期は読めません。
- 窓が広い（幅64）：ゆっくりした波は、周期128のところに帯として出ます。速い波は、30時点しかないのに広い窓でならされて、ほとんど見えなくなります。
- ウェーブレット（幅＝周期）：短い周期には狭い窓、長い周期には広い窓を使います。速い波の時刻と、ゆっくりした波の周期の、両方が読めます。

時刻の細かさと周波数の細かさは、片方を上げると、もう片方が下がります。これを、分解能のトレードオフと呼びます。

灰色のところは、窓が系列の端からはみ出して、計算できないところです。長い周期ほど、広くなります。

ウェーブレット変換は、`WaveletComp`パッケージでも計算できます。同じ系列で比べます。

In [ ]:
wc <- WaveletComp::analyze.wavelet(data.frame(x = mix), "x", loess.span = 0, dt = 1, dj = 1 / 8,
                                   lowerPeriod = 4, upperPeriod = 256,
                                   make.pval = FALSE, verbose = FALSE)
tf_wc <- data.frame(time = rep(t, each = length(wc$Period)),
                    period = rep(wc$Period, times = n),
                    amplitude = as.numeric(wc$Ampl))
plot_tf(tf_wc, breaks = c(6, 16, 32, 64, 128))
# 自分で計算した振幅との相関（自分の計算で NA でないところ）
own <- tf_table(mix, wc$Period, function(p) p)
own <- own[order(own$time, own$period), ]
cor(own$amplitude, tf_wc$amplitude, use = "complete.obs")

模様は同じです。色の目盛りが違うのは、このパッケージが、系列を標準化してから計算し、振幅の尺度も違うからです。
このパッケージは、端のところも値を出します。ただし、端の値は、系列の外を0とみなして計算したもので、信用できません。

## B. 実データを解析する

`tsibbledata`パッケージの`vic_elec`を使います。オーストラリアのビクトリア州の電力需要です。第2回では1日ごとにまとめましたが、今回は30分ごとの値をそのまま使います。

B節は、A節で作った関数（`apply_ma`など）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series     <- "Demand"   # 系列："Demand"（電力需要）か "Temperature"（気温）
year       <- 2014       # 年：2012〜2014
month      <- 6          # 始まりの月：1〜10（その月の1日から）
n_weeks    <- 8          # 期間の長さ（週）：4〜8
ma_len     <- 48         # 移動平均の長さ（点）：48（1日）か 336（1週間）
diff_lag   <- 1          # 差分のラグ（点）：1、48（1日）、336（1週間）のどれか
band_hours <- 24         # 帯域通過フィルタで取り出す周期（時間）：8、12、24 のどれか

### B-1. データを確かめる

In [ ]:
glimpse(vic_elec)
interval(vic_elec)
range(vic_elec$Time)
c(rows = nrow(vic_elec), missing = sum(is.na(vic_elec$Demand)), gaps = has_gaps(vic_elec)$.gaps)

- 1行は30分です。`Demand`が電力需要（MWh）、`Temperature`がメルボルンの気温（℃）です。
- 期間は2012年1月1日から2014年12月31日、52608行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

設定で選んだ月の1日の0時から、`n_weeks`週間分を取り出します。1日は48点、1週間は336点です。
選んだ系列を`y`という名前にします。

In [ ]:
start <- as.POSIXct(sprintf("%d-%02d-01 00:00:00", year, month), tz = "Australia/Melbourne")
n <- n_weeks * 336                                  # 点の数
elec <- vic_elec |>
  filter(Time >= start) |>
  slice(1:n) |>
  mutate(y = .data[[series]])
y_lab <- if (series == "Demand") "Demand (MWh per 30 min)" else "Temperature (deg C)"
range(elec$Time)
c(n = nrow(elec), missing = sum(is.na(elec$y)), gaps = has_gaps(elec)$.gaps)

フィルタも、ピリオドグラムも、等しい間隔で、抜けのない系列を前提にします。間隔が30分で、抜けがないことを確かめました。

このデータには、夏時間の切り替えの日（4月と10月）が6日あり、その日は1日が50点または46点です。ここでは日ごとに切らず、30分ごとの値を連続した系列として使うので、間隔は一定のままで、周期24時間は48点のままです。ただし、切り替えをまたぐ期間を選ぶと、切り替えの前後で、同じ時刻が系列の上では2点（1時間）ずれます。最初の設定（6月から8週間）は、切り替えを含みません。

In [ ]:
ggplot(elec, aes(Time, y)) +
  geom_line() +
  labs(x = "Date", y = y_lab)

### B-3. どの周期があるか

第7回のピリオドグラムで、この系列にどの周期があるかを見ます。横軸は「1日あたりの回数」に直してあります（1が、周期1日）。縦軸は対数の目盛りです。

In [ ]:
pg <- pgram(elec$y) |>
  mutate(per_day = freq * 48,               # 1日あたりの回数
         period_hours = 0.5 / freq)         # 周期（時間）
ggplot(pg |> filter(per_day <= 6), aes(per_day, spec)) +
  geom_line() +
  scale_y_log10() +
  scale_x_continuous(breaks = 0:6) +
  labs(x = "Frequency (cycles per day)", y = "Periodogram (log scale)")
# 強さの大きい順に5つ
peaks <- pg |> arrange(desc(spec)) |> head(5) |>
  mutate(share = spec / sum(pg$spec))     # 全体に占める割合
peaks |> select(period_hours, per_day, share)

### B-4. 移動平均をかける

長さ`ma_len`の移動平均をかけます。

In [ ]:
elec <- elec |> mutate(ma = apply_ma(y, ma_len))
ggplot(elec, aes(Time)) +
  geom_line(aes(y = y), colour = "grey60") +
  geom_line(aes(y = ma), colour = "red", na.rm = TRUE) +
  labs(x = "Date", y = paste(y_lab, "(red: moving average)"))

### B-5. 差分をとる

ラグ`diff_lag`の差分をとります。

In [ ]:
elec <- elec |> mutate(d = apply_diff(y, diff_lag))
ggplot(elec, aes(Time, d)) +
  geom_line(na.rm = TRUE) +
  labs(x = "Date", y = paste("Difference of", y_lab))

### B-6. フィルタは、ねらいどおりに働いたか

まず、理論です。おもな周期（1週間、1日、12時間、8時間、1時間）での、2つのフィルタのゲインを式から出します。

In [ ]:
key <- data.frame(period_hours = c(168, 24, 12, 8, 1))
omega_key <- 0.5 / key$period_hours                  # 1点（30分）あたりの回数
a_ma <- ma_weights(ma_len)
key$gain_ma   <- gain(a_ma, centered_lags(a_ma), omega_key)
key$gain_diff <- gain(c(1, -1), c(0, diff_lag), omega_key)
round(key, 3)

次に、実際の結果です。フィルタをかける前とかけた後のピリオドグラムを重ねます。

In [ ]:
after <- rbind(
  cbind(series = "1 original", pgram(elec$y)),
  cbind(series = "2 moving average", pgram(elec$ma[!is.na(elec$ma)])),
  cbind(series = "3 difference", pgram(elec$d[!is.na(elec$d)]))
)
ggplot(after |> filter(freq * 48 <= 6), aes(freq * 48, spec, colour = series)) +
  geom_line() +
  scale_y_log10() +
  scale_x_continuous(breaks = 0:6) +
  labs(x = "Frequency (cycles per day)", y = "Periodogram (log scale)", colour = NULL)
# 分散：フィルタをかける前と後
var_tab <- c(original = var(elec$y),
             moving_average = var(elec$ma, na.rm = TRUE),
             difference = var(elec$d, na.rm = TRUE))
round(var_tab, 1)

### B-7. 1つの周期を取り出し、振幅の変化を見る

周期`band_hours`時間の成分を、帯域通過フィルタで取り出します。窓の幅は、周期と同じにします。

In [ ]:
band_period <- band_hours * 2                        # 周期を、点の数に直す
z <- band_component(elec$y, band_period)
elec <- elec |> mutate(component = Re(z), amplitude = Mod(z))
ggplot(elec, aes(Time, amplitude)) +
  geom_line(na.rm = TRUE) +
  labs(x = "Date", y = paste("Amplitude of the", band_hours, "hour component"))
# 曜日ごとの、振幅の平均
amp_wday <- elec |>
  as_tibble() |>
  mutate(wday = factor(format(Time, "%a"), levels = c("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"))) |>
  group_by(wday) |>
  summarise(amplitude = mean(amplitude, na.rm = TRUE))
amp_wday

### B-8. スカログラム

周期を4時間から約7.5日まで変えて、ウェーブレット変換をします。縦軸は周期（時間）です。

In [ ]:
periods <- 2^seq(3, 8.5, by = 0.125)                 # 8点（4時間）から約362点（約7.5日）まで
tf <- tf_table(elec$y, periods, function(p) p, time = elec$Time) |>
  mutate(period = period / 2)                        # 周期を、時間に直す
plot_tf(tf, breaks = c(4, 8, 12, 24, 48, 96, 168)) +
  labs(x = "Date", y = "Period (hours)")

読み方：

- B-3：実データの1日の上下は、きれいな波の形をしていません。そのため、周期24時間だけでなく、12時間、8時間のところにも山が出ます。
- B-6：長さ48の移動平均のゲインは、周期24時間、12時間、8時間で、すべて0です。ピリオドグラムでも、その山が消えています。
- B-6：ラグ1の差分は、1日の周期を消しません。周期24時間でのゲインは小さいのですが、0ではないからです。差分のあとのピリオドグラムにも、山が残っています。
- B-6：`diff_lag`を48（季節差分）にすると、周期24時間、12時間、8時間のゲインは0になります。それでも、差分のあとに上下が残ります。1日の形が、毎日まったく同じではないからです（C節で確かめてください）。
- B-7、B-8：振幅は一定ではありません。ピリオドグラムは、期間全体をならした強さしか教えません。スカログラムは、「いつ」強かったかを教えます。
- B-8：灰色のところは計算できません。長い周期を見るには、長い期間が要ります。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"Demand"` `"Temperature"` |
| `year` | 2012〜2014 |
| `month` | 1〜10 |
| `n_weeks` | 4〜8 |
| `ma_len` | 48、336 |
| `diff_lag` | 1、48、336 |
| `band_hours` | 8、12、24 |

読むときの観点：

1. 系列を`"Temperature"`にすると、ピリオドグラムの山はどこに出るか。1週間の周期はあるか。
2. `month`を1（夏）にすると、周期24時間の振幅は、日によってどのくらい変わるか。スカログラムで、振幅の大きい日はいつか。
3. `diff_lag`を1、48、336と変えると、差分のあとのピリオドグラムで、どの山が消え、どの山が残るか。B-6のゲインの表と合っているか。
4. `ma_len`を336にすると、移動平均のあとに何が残るか。`n_weeks`を4にすると、スカログラムの灰色のところはどうなるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（Demand、2014年6月から8週間）の結果と比べてください。